# PyTorch: tensores y diferenciación automática

PyTorch es una biblioteca de código abierto para trabajar con **tensores** y construir modelos de aprendizaje automático. Su API resulta familiar si ya se conocen los arrays de NumPy, pero añade herramientas especialmente útiles para la optimización:

- **Tensores:** estructuras multidimensionales para almacenar y transformar datos.
- **Autograd:** sistema de diferenciación automática que calcula gradientes a partir de las operaciones realizadas.
- **Grafos dinámicos:** las operaciones se registran mientras se ejecuta el programa, lo que facilita experimentar y modificar los modelos.

En este resumen primero se realizan operaciones básicas con tensores y después se calcula la derivada de una función sencilla. Esta misma idea aparece al entrenar modelos: se calcula una función de coste, se obtiene su gradiente y se actualizan los parámetros para mejorar el resultado.

PyTorch también puede combinarse con herramientas científicas como NumPy y SciPy. En computación cuántica, sus gradientes son especialmente interesantes para optimizar parámetros de circuitos variacionales junto con Qiskit.

In [ ]:
# Iniciación de librerías
import torch
import numpy as np
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2

## 1.1. Tensores (`torch.Tensor`)

Un tensor es una estructura multidimensional que almacena datos numéricos (en CPU o GPU) y permite registrar operaciones para el cálculo de gradientes.

Es el equivalente de:
- **Un escalar (dimensión 0):** un valor único (ej. el valor de la función de pérdida o *loss*).
- **Un vector (dimensión 1):** una lista de valores (ej. las variables clínicas de un paciente o los parámetros de un circuito).
- **Una matriz (dimensión 2):** una tabla de datos (ej. un conjunto tabular de forma `[n_pacientes, n_variables]`).
- **Un tensor de dimensión superior (dimensión 3 o más):** datos estructurados como señales temporales fisiológicas ECG/EEG (`[batch, canales, tiempo]`), imágenes médicas 2D (`[batch, canales, alto, ancho]`) o volúmenes 3D como resonancias magnéticas (`[batch, canales, profundidad, alto, ancho]`).

In [ ]:
# Escalar (0D)
x = torch.tensor(5.0)

# Vector (1D)
v = torch.tensor([1.0, 2.0, 3.0])

# Matriz (2D)
m = torch.tensor([
    [1.0, 2.0],
    [3.0, 4.0]
])

# Tensor tridimensional (3D)
t = torch.randn(2, 3, 4)

print(x.shape)  # torch.Size([])
print(v.shape)  # torch.Size([3])
print(m.shape)  # torch.Size([2, 2])
print(t.shape)  # torch.Size([2, 3, 4])

### Resumen de operaciones con tensores

| Operación | Función / Atributo | Aplicación en análisis de datos biomédicos |
| :--- | :--- | :--- |
| **Crear tensores** | `torch.tensor()`, `torch.zeros()`, `torch.ones()` | Inicializar datos tabulares, máscaras o parámetros. |
| **Tensores aleatorios** | `torch.rand()`, `torch.randn()` | Inicializar pesos o generar lotes sintéticos de prueba. |
| **Dimensiones** | `.shape`, `.ndim`, `.numel()` | Comprobar tamaños de *batch*, número de pacientes y variables. |
| **Tipo de dato y dispositivo** | `.dtype`, `.device`, `.to()` | Gestionar precisión (`float32`, `float64`) y ejecución en CPU/GPU. |
| **Convertir tipos** | `.float()`, `.long()`, `.bool()` | Variables continuas en `float32`; etiquetas de clase en `long` (`int64`). |
| **Cambiar dimensiones** | `.reshape()`, `.view()`, `.unsqueeze()`, `.squeeze()` | Aplanar entradas o añadir la dimensión de *batch* (`[1, n_features]`). |
| **Transponer / Reordenar** | `.T`, `.transpose()`, `.permute()` | Reordenar ejes en matrices, señales (`[B, C, T]`) o imágenes (`[B, C, H, W]`). |
| **Concatenar** | `torch.cat()` | Unir cohortes de pacientes (`dim=0`) o añadir variables (`dim=1`). |
| **Apilar** | `torch.stack()` | Agrupar muestras en un *batch* o cortes 2D en un volumen 3D. |
| **Indexar y filtrar** | `tensor[i]`, máscaras booleanas | Seleccionar pacientes, columnas o filtrar valores anómalos y `NaN`. |
| **Operaciones matriciales** | `torch.matmul()`, `@` | Transformaciones lineales ($X W + b$) y proyecciones de características. |
| **Estadísticas y reducción** | `.mean()`, `.std()`, `.argmax()` | Estandarización *Z-score* (`dim=0`) y selección de la clase predicha (`dim=1`). |
| **Conversión con NumPy** | `torch.from_numpy()`, `.detach().cpu().numpy()` | Integración con `pandas`, `scikit-learn` y `qiskit`. |

### Creación, tipos de datos y dispositivo

- **Inicialización:** además de `torch.tensor()`, se emplean `torch.zeros()`, `torch.ones()`, `torch.rand()` (distribución uniforme en $[0, 1)$) y `torch.randn()` (distribución normal $\mathcal{N}(0, 1)$).
- **Tipos de datos (`.dtype`):**
  - `torch.float32` (`.float()`): estándar para variables continuas (biomarcadores, señales, imágenes) y pesos de modelos.
  - `torch.int64` (`.long()`): requerido para etiquetas de clasificación multiclase (ej. en `nn.CrossEntropyLoss`).
  - `torch.bool` (`.bool()`): empleado en máscaras de filtrado.
- **Dispositivo (`.device`):** los tensores que operan juntos deben residir en el mismo dispositivo (`cpu` o `cuda`).

In [ ]:
torch.manual_seed(42)  # Semilla para reproducibilidad

# Matriz de 4 pacientes x 3 biomarcadores [Edad, Glucosa, Colesterol]
datos = torch.tensor([
    [45.0,  92.0, 180.0],
    [67.0, 155.0, 240.0],
    [52.0, 130.0, 210.0],
    [34.0,  85.0, 165.0]
])

# Etiquetas diagnósticas (0: Sano, 1: Patológico)
etiquetas = torch.tensor([0, 1, 1, 0]).long()

# Propiedades y paso a GPU si está disponible
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
datos = datos.to(device)

print("Shape:", datos.shape, "| ndim:", datos.ndim, "| dtype:", datos.dtype, "| device:", datos.device)
print("Etiquetas:", etiquetas, "| dtype:", etiquetas.dtype)

### Indexación, filtrado y tratamiento de `NaN`

El acceso por índices sigue la sintaxis `tensor[filas, columnas]`. Mediante condiciones lógicas se generan máscaras booleanas para filtrar subgrupos clínicos o imputar valores faltantes (`NaN`).

In [ ]:
# Selección de filas y columnas
primer_paciente = datos[0]        # Fila 0
columna_glucosa = datos[:, 1]     # Columna 1 de todos los pacientes

# Filtrado por máscara booleana (pacientes con glucosa > 120)
mascara = datos[:, 1] > 120.0
pacientes_filtrados = datos[mascara]

# Detección y reemplazo de valores faltantes (NaN) por la media
muestra = torch.tensor([1.2, float('nan'), 3.5, 2.8])
media_valida = muestra[~torch.isnan(muestra)].mean()
muestra_imputada = torch.where(torch.isnan(muestra), media_valida, muestra)

print("Glucosa:", columna_glucosa)
print("Pacientes con glucosa > 120:\n", pacientes_filtrados)
print("Muestra sin NaN:", muestra_imputada)

### Cambio de dimensiones y transposición

- **`.reshape()` y `.view()`:** modifican las dimensiones manteniendo el número total de elementos. El valor `-1` infiere automáticamente el tamaño de esa dimensión. `.view()` exige que el tensor sea contiguo en memoria, mientras que `.reshape()` crea una copia si es necesario.
- **`.unsqueeze(dim)` y `.squeeze(dim)`:** añaden o eliminan una dimensión de tamaño 1 (útil para convertir una muestra `[n_features]` en un lote `[1, n_features]`).
- **`.T`, `.transpose(dim0, dim1)` y `.permute(...)`:** `.T` transpone matrices 2D; `.transpose()` intercambia dos ejes; `.permute()` reordena todas las dimensiones.

In [ ]:
# Aplanado de un lote de 4 imágenes de 1x8x8 a vectores de 64 elementos
imagenes = torch.randn(4, 1, 8, 8)
aplanadas = imagenes.reshape(4, -1)  # Forma: [4, 64]

# Añadir y quitar dimensión de batch en una muestra individual
muestra_individual = datos[0]                       # Forma: [3]
muestra_batch = muestra_individual.unsqueeze(0)     # Forma: [1, 3]
muestra_recuperada = muestra_batch.squeeze(0)       # Forma: [3]

# Transposición de matriz 2D (.T) e intercambio de ejes en señales 3D (.transpose)
datos_T = datos.T                                   # De [4, 3] a [3, 4]
eeg = torch.randn(16, 500, 19)                      # [batch, tiempo, canales]
eeg_canales_primero = eeg.transpose(1, 2)           # [batch, canales, tiempo] -> [16, 19, 500]

print("Imágenes aplanadas:", aplanadas.shape)
print("Muestra con batch:", muestra_batch.shape)
print("Datos transpuestos (.T):", datos_T.shape)
print("EEG reordenado:", eeg_canales_primero.shape)

### Unión de tensores: `torch.cat()` y `torch.stack()`

- **`torch.cat([t1, t2], dim=...)`:** concatena tensores a lo largo de una dimensión existente (`dim=0` añade filas/pacientes; `dim=1` añade columnas/variables).
- **`torch.stack([t1, t2], dim=...)`:** apila tensores del mismo tamaño creando una dimensión nueva.

In [ ]:
bloque_1 = torch.tensor([[1.0, 2.0], [3.0, 4.0]])  # Forma: [2, 2]
bloque_2 = torch.tensor([[5.0, 6.0], [7.0, 8.0]])  # Forma: [2, 2]

concat_filas = torch.cat([bloque_1, bloque_2], dim=0)     # Forma: [4, 2]
concat_columnas = torch.cat([bloque_1, bloque_2], dim=1)  # Forma: [2, 4]
apilados = torch.stack([bloque_1, bloque_2], dim=0)       # Forma: [2, 2, 2]

print("cat(dim=0):", concat_filas.shape)
print("cat(dim=1):", concat_columnas.shape)
print("stack(dim=0):", apilados.shape)

### Operaciones matriciales, normalización y conversión con NumPy

- **Producto elemento a elemento (`*`) vs producto matricial (`torch.matmul()` o `@`):** el producto matricial se utiliza en las capas lineales ($Y = X W + b$).
- **Reducciones por eje:** funciones como `.mean(dim=0)` o `.std(dim=0)` calculan estadísticos por variable para estandarizar los datos ($Z = \frac{X - \mu}{\sigma}$), mientras que `.argmax(dim=1)` devuelve el índice de la clase con mayor puntuación por muestra.
- **Interoperabilidad con NumPy:** `torch.from_numpy()` convierte un array en tensor y `.detach().cpu().numpy()` realiza el paso inverso.

In [ ]:
# 1. Estandarización Z-score por columna (dim=0)
media = datos.mean(dim=0, keepdim=True)
std = datos.std(dim=0, keepdim=True)
datos_norm = (datos - media) / (std + 1e-8)

# 2. Producto matricial (transformación lineal de 3 variables a 2 salidas)
pesos = torch.randn(3, 2, device=device)
sesgo = torch.zeros(2, device=device)
salida = torch.matmul(datos_norm, pesos) + sesgo  # Equivalente a: datos_norm @ pesos + sesgo
prediccion = salida.argmax(dim=1)

# 3. Conversión entre PyTorch y NumPy
array_np = datos_norm.detach().cpu().numpy()
tensor_desde_np = torch.from_numpy(array_np).float()

print("Datos estandarizados:\n", datos_norm)
print("Salida lineal [4, 2]:\n", salida)
print("Clases predichas:", prediccion)

---

## 1.2. Autograd y Gestión del Grafo Computacional

El motor **Autograd** de PyTorch proporciona diferenciación automática sobre todas las operaciones realizadas con tensores. A diferencia de los frameworks de grafo estático, PyTorch construye un **grafo computacional dinámico** (*define-by-run*): el **DAG (*Directed Acyclic Graph* / Grafo Acíclico Dirigido)** se genera al vuelo durante la ejecución hacia adelante (*forward pass*) y se recorre en sentido inverso durante la retropropagación (*backward pass*).

En este grafo:
- Los **nodos hoja (*leaf tensors*)** son los tensores creados directamente por el usuario (como los pesos y sesgos del modelo o los parámetros variacionales $\boldsymbol{\theta}$ de un circuito cuántico), con atributo `is_leaf = True`.
- Los **nodos intermedios y de salida (raíces)** son el resultado de operaciones matemáticas entre tensores y almacenan referencias a las funciones de derivación necesarias para aplicar la regla de la cadena.

### Mecánica de gradientes (`requires_grad`)

- **`x = torch.tensor(..., requires_grad=True)`:** activa el rastreo de operaciones sobre el tensor `x` para construir el DAG. Cualquier tensor derivado de una operación que involucre a `x` heredará automáticamente `requires_grad=True`.
- **`.grad_fn`:** atributo que actúa como apuntador a la operación matemática que generó el tensor en el grafo (por ejemplo, `<AddBackward0>` para una suma, `<MulBackward0>` para un producto o `<MeanBackward0>` para una media). Los tensores hoja creados por el usuario tienen `grad_fn = None`.
- **`.backward()`:** inicia la retropropagación desde el tensor escalar sobre el que se invoca (habitualmente la pérdida $\mathcal{L}$ o una salida escalar $y$). Ejecuta la **regla de la cadena hacia atrás**, calculando las derivadas parciales $\frac{\partial y}{\partial x}$ con respecto a todos los tensores hoja con `requires_grad=True` y almacenando/acumulando el resultado en el atributo **`x.grad`**.
  - *Nota:* `.backward()` requiere por defecto que el tensor de partida sea un **escalar** (dimensión 0). Si se invoca sobre un vector o matriz, es necesario pasar un tensor de pesos del mismo tamaño (`y.backward(gradient=v)`) para calcular el producto vector-Jacobiano ($v^T J$).
- **Peligro de acumulación de gradientes:** PyTorch **suma** los nuevos gradientes a los valores ya existentes en `x.grad` por defecto ($g_{\text{total}} \leftarrow g_{\text{anterior}} + \nabla_x y$). Aunque este diseño es útil en técnicas avanzadas como la *acumulación de gradientes* (simular un *batch size* mayor cuando la VRAM es limitada) o en arquitecturas recurrentes, en el entrenamiento estándar **requiere limpiar explícitamente los gradientes con `optimizer.zero_grad()` (o `x.grad.zero_()`) en cada iteración** para evitar contaminar la actualización actual con gradientes de lotes pasados.

### Control del grafo y ahorro de memoria

- **`with torch.no_grad():`:** gestor de contexto (también utilizable como decorador `@torch.no_grad()`) que **deshabilita el seguimiento del grafo computacional** para todas las operaciones ejecutadas en su bloque. Durante la **inferencia, validación o evaluación**, no se necesita ejecutar `.backward()`, por lo que almacenar las activaciones intermedias del grafo desperdiciaría memoria. Usar `with torch.no_grad():` **reduce drásticamente el consumo de VRAM** y **acelera la ejecución**. *(Alternativa moderna para inferencia pura: `with torch.inference_mode():`, aún más rápida al desactivar también contadores de versión internos).* 
- **`.detach()`:** corta un tensor del grafo computacional actual devolviendo una **nueva vista que comparte la misma memoria subyacente pero sin historial de gradiente** (`requires_grad=False`, `grad_fn=None`). Es **imprescindible** cuando se necesita:
  1. Pasar activaciones intermedias, predicciones o pérdidas a librerías externas (NumPy, Pandas, métricas como `roc_auc_score` de Scikit-Learn, o rutinas de visualización con Matplotlib) sin romper el grafo original ni provocar un `RuntimeError: Can't call numpy() on Tensor that requires grad`.
  2. Utilizar la salida de una subred como entrada fija para otra sin propagar gradientes hacia la primera.

### Modificación directa *in-place* de parámetros

- En PyTorch, los métodos terminados en guion bajo (como `.add_()`, `.sub_()`, `.zero_()` o asignaciones `x -= lr * x.grad`) modifican el contenido del tensor directamente en su dirección de memoria (*in-place*) sin reservar un tensor nuevo.
- Aplicar una operación *in-place* directamente sobre un tensor hoja con `requires_grad=True` dentro del grafo activo lanza un error (`RuntimeError: a leaf Variable that requires grad is being used in an in-place operation`) o corrompe los valores necesarios para el cálculo de derivadas.
- **Solución recomendada:** envolver cualquier modificación manual de pesos dentro de un bloque **`with torch.no_grad():`** para actualizar los parámetros sin que Autograd registre dicha operación en el historial del grafo. *(Históricamente se accedía a `tensor.data`, pero hoy se desaconseja en favor de `with torch.no_grad():` porque `tensor.data` elude el control de versiones de seguridad de Autograd).*

| Mecanismo / Atributo | Propósito principal | Efecto sobre el grafo (DAG) y memoria |
| :--- | :--- | :--- |
| `requires_grad=True` | Marcar un tensor para calcular derivadas respecto a él | Registra operaciones posteriores y retiene activaciones en memoria para `.backward()`. |
| `.grad_fn` | Inspeccionar la operación que creó un tensor derivado | Apunta al nodo del grafo inverso (ej. `<AddBackward0>`); es `None` en nodos hoja. |
| `.backward()` | Ejecutar la regla de la cadena ($\frac{\partial y}{\partial x}$) | Recorre el DAG hacia atrás, libera los búferes intermedios y **acumula** en `.grad`. |
| `optimizer.zero_grad()` | Reiniciar los gradientes acumulados a cero | Evita que los gradientes de iteraciones previas se sumen a la iteración actual. |
| `with torch.no_grad():` | Evaluar/inferir o actualizar pesos manualmente | Desactiva la creación del DAG en ese bloque; ahorra VRAM y acelera el cómputo. |
| `.detach()` | Desvincular un tensor específico del grafo | Devuelve una vista sin gradiente (`requires_grad=False`) segura para Scikit-Learn / NumPy. |

In [ ]:
# 1. Construcción del DAG y cálculo analítico vs automático de gradientes
# Sea y = (w * x + b)^2 con x = 2.0, w = 3.0, b = 1.0
# Derivadas analíticas: u = w*x + b = 7 -> dy/dw = 2*u*x = 28, dy/db = 2*u = 14
x = torch.tensor(2.0, requires_grad=False)  # Entrada fija (no requiere gradiente)
w = torch.tensor(3.0, requires_grad=True)   # Parámetro entrenable (nodo hoja)
b = torch.tensor(1.0, requires_grad=True)   # Sesgo entrenable (nodo hoja)

u = w * x + b       # Operación intermedia
y = u ** 2          # Salida escalar

print("¿'w' es nodo hoja (is_leaf)?:", w.is_leaf, "| w.grad_fn:", w.grad_fn)
print("¿'u' es nodo hoja (is_leaf)?:", u.is_leaf, "| u.grad_fn:", u.grad_fn)
print("Función generadora de 'y' (y.grad_fn):", y.grad_fn)

# Ejecución de la regla de la cadena hacia atrás
y.backward()
print(f"Gradientes tras 1er backward -> dy/dw: {w.grad.item():.1f} | dy/db: {b.grad.item():.1f}")

# 2. Demostración del peligro de acumulación de gradientes
y_nuevo = (w * x + b) ** 2
y_nuevo.backward()  # ¡Sin limpiar gradientes, PyTorch SUMA al valor anterior!
print(f"Gradientes ACUMULADOS (sin zero_grad) -> dy/dw: {w.grad.item():.1f} (28 + 28)")

# Limpieza correcta antes de un nuevo cálculo
w.grad.zero_()
b.grad.zero_()
y_limpio = (w * x + b) ** 2
y_limpio.backward()
print(f"Gradientes tras zero_() -> dy/dw: {w.grad.item():.1f} | dy/db: {b.grad.item():.1f}")

# 3. Uso de .detach() para exportar activaciones y with torch.no_grad() para actualización in-place
activacion_sin_grafo = u.detach()  # Vista compartida pero cortada del DAG
print("activacion_sin_grafo.requires_grad:", activacion_sin_grafo.requires_grad)

lr = 0.01
with torch.no_grad():  # Modificación in-place segura sin corromper Autograd
    w -= lr * w.grad
    b -= lr * b.grad

print(f"Parámetros actualizados manualmente -> w: {w.item():.4f} | b: {b.item():.4f}")

---

## 1.3. Canalización Eficiente: `Dataset` y `DataLoader`

En problemas biomédicos reales —como el análisis de volúmenes médicos 3D (resonancia magnética NIfTI, tomografía computarizada DICOM), imágenes histopatológicas de ultra-alta resolución (*Whole Slide Images*) o registros prolongados de EEG/ECG multicanal— **cargar todos los arrays simultáneamente en memoria RAM resulta inviable** y provocaría el colapso del sistema. 

Para desacoplar el almacenamiento en disco del entrenamiento en GPU, PyTorch formaliza el pipeline de datos mediante dos abstracciones dentro de `torch.utils.data`:
1. **`Dataset`:** define **cómo acceder a una muestra individual** y su etiqueta a partir de su índice (aplicando carga perezosa o *lazy loading*).
2. **`DataLoader`:** envuelve al `Dataset` en un iterador eficiente que gestiona la **agrupación por lotes (*batching*), mezcla aleatoria (*shuffling*), multiprocesamiento asíncrono en CPU y transferencia optimizada a GPU**.

### Estructura de un `torch.utils.data.Dataset` personalizado

Cualquier clase personalizada debe heredar de `Dataset` y sobrescribir obligatoriamente tres métodos mágicos:
- **`__init__(self, file_paths, labels, transform=None)`:** se ejecuta una única vez al instanciar el dataset. **Solo debe almacenar metadatos ligeros** en memoria (lista de rutas en disco, vector de etiquetas clínicas, diccionario de metadatos y el pipeline de transformaciones/aumentado de datos `transform`), nunca cargar todas las imágenes o volúmenes.
- **`__len__(self)`:** devuelve el número total de muestras disponibles en la cohorte (`len(self.file_paths)`).
- **`__getitem__(self, idx)`:** implementa la **carga perezosa (*lazy loading*)**. Lee del disco duro únicamente el archivo correspondiente al índice `idx` en el instante en que el `DataLoader` lo solicita, aplica las transformaciones (`self.transform`) al vuelo y devuelve los tensores con los tipos exigidos por PyTorch (`torch.float32` para las características de entrada $x$ y `torch.long` para etiquetas de clasificación $y$).

```python
from torch.utils.data import Dataset

class MedicalDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Carga perezosa (lazy loading): lee de disco solo cuando se pide el índice
        x = load_image_or_volume(self.file_paths[idx])
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)
```

### Parámetros críticos de `DataLoader`

| Parámetro | Valor típico | Función técnica e impacto en el pipeline biomédico |
| :--- | :--- | :--- |
| **`batch_size`** | `16`, `32`, `64` | Tamaño del lote procesado en paralelo. Un valor mayor estabiliza la estimación del gradiente y aprovecha mejor la GPU, pero consume más VRAM. |
| **`shuffle`** | `True` (train) / `False` (val/test) | **Mezcla aleatoria en cada época.** Se activa **solo en entrenamiento** (`shuffle=True`) para romper correlaciones temporales, de centro hospitalario o de orden de captura. En validación y test debe ser `False` para mantener el orden reproducible de evaluación. |
| **`num_workers`** | `0`, `2`, `4`, `8` | Número de **procesos paralelos en CPU** dedicados a la precarga asíncrona de lotes desde disco y aplicación de `transform`. Evita que la GPU quede ociosa esperando lecturas de disco (*cuello de botella I/O*). *(En Windows/Jupyter interactivo a veces se usa `0` para depurar en el proceso principal).* |
| **`pin_memory`** | `True` (con GPU) | Reserva los tensores del lote en **memoria RAM bloqueada (*page-locked / pinned memory*)**, impidiendo que el sistema operativo la pagine a disco y habilitando transferencias directas **CPU $\rightarrow$ GPU mucho más rápidas a través del bus PCIe** mediante DMA (*Direct Memory Access*). |
| **`collate_fn`** | Función personalizada | Función que recibe una lista de tuplas `[(x_0, y_0), ..., (x_B, y_B)]` devueltas por `__getitem__` y las ensambla en un único lote. Es **obligatoria si las muestras tienen dimensiones variables** (ej. señales ECG de distinta duración que requieren *padding*, volúmenes con distinto número de cortes axiales o grafos moleculares) o formatos no rectangulares antes del apilado (`torch.stack`). |
| **`drop_last`** | `True` / `False` | Si el total de muestras no es divisible por `batch_size`, descarta el último lote incompleto (útil en entrenamiento si el modelo usa `BatchNorm1d`/`2d`, que falla si el último lote queda de tamaño `1`). |

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Simulación de función de lectura desde disco (ej. lectura de corte MRI o señal biomédica)
def load_image_or_volume(file_path):
    # En un caso real: nib.load(file_path).get_fdata() o np.load(file_path)
    rng = np.random.default_rng(abs(hash(file_path)) % (2**32))
    return rng.normal(loc=0.0, scale=1.0, size=(1, 16, 16)).astype(np.float32)

class MedicalDataset(Dataset):
    def __init__(self, file_paths, labels, transform=None):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        # Carga perezosa (lazy loading): lee de disco solo cuando se pide el índice
        x = load_image_or_volume(self.file_paths[idx])
        y = self.labels[idx]
        if self.transform:
            x = self.transform(x)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

# Ejemplo de collate_fn personalizada (apila tensores y filtra o rellena si fuera necesario)
def custom_collate_fn(batch):
    # batch es una lista de tuplas: [(x_0, y_0), (x_1, y_1), ...]
    inputs, targets = zip(*batch)
    inputs_stacked = torch.stack(inputs, dim=0)
    targets_stacked = torch.stack(targets, dim=0)
    return inputs_stacked, targets_stacked

# Creación de rutas simuladas y etiquetas para entrenamiento y validación
rutas_train = [f"paciente_train_{i:03d}.nii.gz" for i in range(20)]
labels_train = [i % 2 for i in range(20)]

train_dataset = MedicalDataset(rutas_train, labels_train, transform=lambda img: (img - img.mean()) / (img.std() + 1e-8))

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=8,
    shuffle=True,                         # Mezcla aleatoria activa en entrenamiento
    num_workers=0,                        # En scripts de producción usar p.ej. num_workers=4
    pin_memory=torch.cuda.is_available(), # Memoria bloqueada para transferencia rápida CPU -> GPU
    collate_fn=custom_collate_fn
)

# Iteración sobre el primer lote
lote_x, lote_y = next(iter(train_loader))
print("Total de muestras en el Dataset:", len(train_dataset))
print("Forma del lote de entrada [batch, C, H, W]:", lote_x.shape, "| dtype:", lote_x.dtype)
print("Forma del lote de etiquetas [batch]:", lote_y.shape, "| dtype:", lote_y.dtype)

---

## 1.4. Construcción Modular de Modelos (`torch.nn`)

El paquete `torch.nn` proporciona los bloques de construcción para diseñar redes neuronales profundas y arquitecturas híbridas cuántico-clásicas (donde un circuito cuántico parametrizado de Qiskit/PennyLane se integra como una capa más dentro de un módulo de PyTorch).

### Estructura base heredando de `nn.Module`

Todo modelo o subcapa personalizada debe heredar de `nn.Module` y definir dos métodos fundamentales:
1. **`__init__(self, ...)`:** debe invocar obligatoriamente `super().__init__()` en la primera línea para inicializar el registro interno de parámetros, *buffers* y *hooks*. Aquí se instancian las capas con pesos entrenables o bloques secuenciales (`nn.Sequential`).
2. **`forward(self, x)`:** define la transformación matemática que experimenta el tensor de entrada `x` hasta producir la salida. **Importante:** al usar el modelo, siempre se invoca como `outputs = model(x)` y **nunca** `model.forward(x)` directamente, ya que `__call__` gestiona internamente los *hooks* del grafo y las utilidades de PyTorch. Además, en tareas de clasificación la última capa debe **devolver los *logits* crudos** (sin función de activación `Softmax`), pues la función de coste los procesará de forma numéricamente estable.

```python
import torch.nn as nn

class NeuralNetwork(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3)
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)  # Devolver logits sin softmax
```

### Capas fundamentales en `torch.nn`

- **Capas Lineales (Densas):**
  - `nn.Linear(in_features, out_features, bias=True)`: aplica la proyección afín $Y = X W^T + b$, donde $W \in \mathbb{R}^{\text{out} \times \text{in}}$ y $b \in \mathbb{R}^{\text{out}}$. Esencial en datos tabulares clínicos, cuellos de botella latentes (antes de codificar características en qubits) y cabezales de clasificación.
- **Capas de Convolución:**
  - `nn.Conv1d` (señales fisiológicas temporales ECG/EEG), `nn.Conv2d` (imágenes médicas 2D como rayos X o dermatoscopia) y `nn.Conv3d` (volúmenes tomográficos 3D de RM/TC).
  - Definidas por sus hiperparámetros geométricos: `in_channels` (canales de entrada, ej. `1` en escala de grises o `3` en RGB), `out_channels` (número de filtros o mapas de características aprendidos), `kernel_size` (tamaño de la ventana receptiva), `stride` (paso de desplazamiento) y `padding` (relleno en los bordes para conservar las dimensiones espaciales).
- **Capas de Pooling (Submuestreo espacial):**
  - `nn.MaxPool2d(kernel_size, stride)` / `nn.AvgPool2d`: reducen la dimensionalidad espacial seleccionando el máximo o el promedio local, aportando invariancia a pequeñas traslaciones.
  - **`nn.AdaptiveAvgPool2d((1, 1))`** (o `nn.AdaptiveAvgPool3d((1, 1, 1))`): calcula automáticamente el *kernel* y el *stride* necesarios para que cualquier mapa de características de tamaño $[B, C, H, W]$ se reduzca exactamente a $[B, C, 1, 1]$. **Ventaja clave:** permite que el clasificador lineal posterior (`nn.Linear(C, num_classes)`) admita **imágenes de entrada con resolución espacial arbitraria** sin que fallen las dimensiones matriciales.
- **Capas de Normalización:**
  - `nn.BatchNorm1d(num_features)` / `nn.BatchNorm2d(num_channels)` / `nn.BatchNorm3d(num_channels)`: normalizan las activaciones de cada canal restando la media y dividiendo por la desviación típica del lote actual, incorporando además dos parámetros aprendibles ($\gamma$ de escala y $\beta$ de desplazamiento). Durante el entrenamiento mantienen medias y varianzas móviles (*running_mean*, *running_var*) que luego se fijan y utilizan en validación/inferencia.
  - `nn.LayerNorm(normalized_shape)`: normaliza a lo largo de las características de cada muestra individual en lugar de a través del lote. Resulta **clave para la estabilidad y convergencia en redes con gradientes profundos**, arquitecturas *Transformer* o cuando el tamaño de lote (*batch size*) es muy pequeño (frecuente en volúmenes médicos 3D por límites de VRAM).
- **Capas de Regularización:**
  - `nn.Dropout(p=0.5)` (o `nn.Dropout2d`): durante el entrenamiento (`model.train()`), anula aleatoriamente cada activación con probabilidad $p$ y escala las restantes por $\frac{1}{1-p}$ (*inverted dropout*), evitando la coadaptación de neuronas y previniendo el **sobreajuste (*overfitting*)** en cohortes médicas reducidas. En evaluación (`model.eval()`), se desactiva automáticamente actuando como la función identidad.

### Inspección e inicialización de pesos

- **`model.parameters()` y `model.named_parameters()`:** devuelven iteradores sobre los tensores entrenables (`nn.Parameter`) del modelo. `named_parameters()` proporciona tuplas `(nombre, tensor)` (ej. `'feature_extractor.0.weight'`), muy útiles para auditar gradientes, aplicar tasas de aprendizaje diferenciadas por capa o congelar pesos en *Transfer Learning* (`param.requires_grad = False`).
- **Conteo de parámetros entrenables:**
  ```python
  total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
  ```
- **Inicialización de pesos (`torch.nn.init`):** una mala inicialización provoca que la varianza de las activaciones y de los gradientes crezca exponencialmente (**explosión de gradientes**) o tienda a cero (**desvanecimiento de gradientes**) al atravesar múltiples capas:
  - **Xavier / Glorot (`nn.init.xavier_uniform_`, `nn.init.xavier_normal_`):** mantiene la varianza constante entre la entrada y la salida de la capa ($\text{Var}(W) = \frac{2}{n_{\text{in}} + n_{\text{out}}}$). Es la elección idónea para capas seguidas de funciones de activación simétricas o diferenciables en el origen como `Tanh`, `Sigmoid` o proyecciones lineales hacia ángulos de rotación cuántica ($[-\pi, \pi]$).
  - **He / Kaiming (`nn.init.kaiming_normal_`, `nn.init.kaiming_uniform_`):** ajusta la varianza a $\text{Var}(W) = \frac{2}{n_{\text{in}}}$ para compensar que las activaciones rectificadas (`ReLU`, `LeakyReLU`) anulan la mitad negativa de la distribución.

In [ ]:
import torch.nn as nn

# 1. Modelo tabular modular heredando de nn.Module
class NeuralNetwork(nn.Module):
    def __init__(self, in_features, num_classes):
        super().__init__()
        self.feature_extractor = nn.Sequential(
            nn.Linear(in_features, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3)
        )
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, x):
        features = self.feature_extractor(x)
        return self.classifier(features)  # Devolver logits sin softmax

# 2. Modelo convolucional 2D con AdaptiveAvgPool2d((1, 1)) para resolución arbitraria
class MedicalCNN(nn.Module):
    def __init__(self, in_channels=1, num_classes=2):
        super().__init__()
        self.conv_block = nn.Sequential(
            nn.Conv2d(in_channels, 16, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))  # Salida garantizada de [batch, 32, 1, 1]
        )
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(32, num_classes)

    def forward(self, x):
        x = self.conv_block(x)
        x = x.flatten(start_dim=1)        # De [batch, 32, 1, 1] a [batch, 32]
        x = self.dropout(x)
        return self.fc(x)                 # Logits crudos

# Función para inicializar pesos con He/Kaiming en capas con ReLU y Xavier en el clasificador
def init_weights(m):
    if isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        nn.init.zeros_(m.bias)

modelo_tabular = NeuralNetwork(in_features=10, num_classes=2)
modelo_cnn = MedicalCNN(in_channels=1, num_classes=2)
modelo_cnn.apply(init_weights)

# Inspección de parámetros entrenables
num_params_tabular = sum(p.numel() for p in modelo_tabular.parameters() if p.requires_grad)
num_params_cnn = sum(p.numel() for p in modelo_cnn.parameters() if p.requires_grad)
print(f"Parámetros entrenables -> NeuralNetwork: {num_params_tabular} | MedicalCNN: {num_params_cnn}")

for name, param in modelo_tabular.named_parameters():
    print(f"  {name:30s} | shape: {str(list(param.shape)):12s} | requires_grad: {param.requires_grad}")

# Comprobación de que MedicalCNN admite resoluciones de imagen distintas gracias a AdaptiveAvgPool2d
img_16x16 = torch.randn(4, 1, 16, 16)
img_64x64 = torch.randn(4, 1, 64, 64)
print("Salida CNN con entrada 16x16:", modelo_cnn(img_16x16).shape, "| con 64x64:", modelo_cnn(img_64x64).shape)

---

## 1.5. Pérdidas, Optimización y Planificación de Tasa de Aprendizaje

Una vez definida la arquitectura, el aprendizaje consiste en minimizar una **función de pérdida (coste)** $\mathcal{L}(\boldsymbol{\theta})$ ajustando iterativamente los parámetros mediante un **optimizador** y modulando el paso de actualización con un **planificador de tasa de aprendizaje (*LR Scheduler*)**.

### Funciones de coste (`torch.nn`) y sus peculiaridades

- **`nn.CrossEntropyLoss()` (Clasificación multiclase o binaria con 2 salidas):**
  - **Aviso crítico:** En PyTorch, `nn.CrossEntropyLoss()` **combina internamente `nn.LogSoftmax(dim=1)` y `nn.NLLLoss()`** en una única expresión analítica que emplea el truco *Log-Sum-Exp* para evitar errores de redondeo o desbordamiento numérico (*overflow/underflow* al calcular $e^{z_i}$).
  - **Requisitos obligatorios:**
    1. La última capa del modelo debe entregar **logits crudos** de dimensión `[batch, num_classes]`, **sin aplicar una capa `Softmax` previa** (aplicar Softmax antes distorsionaría los gradientes y frenaría el aprendizaje).
    2. Las etiquetas verdaderas `y` deben ser **enteros de índice de clase de tipo `torch.long` (`int64`)** con dimensión `[batch]` y valores en $\{0, 1, \dots, C-1\}$ (no vectores *one-hot*).
  - *Aplicación biomédica:* admite el parámetro `weight=torch.tensor([w_0, w_1, ...])` para penalizar más los errores en clases patológicas minoritarias en cohortes desbalanceadas.
- **`nn.BCEWithLogitsLoss()` (Clasificación binaria de 1 salida o multietiqueta):**
  - Combina internamente una capa **`Sigmoid` ($\sigma(z) = \frac{1}{1+e^{-z}}$) con la entropía cruzada binaria (`nn.BCELoss`)** logrando una **estabilidad numérica superior** frente a valores extremos de los logits.
  - **Requisitos obligatorios:** la salida del modelo deben ser **logits crudos** (sin `Sigmoid`) de forma `[batch, 1]` o `[batch, num_etiquetas]`, y las etiquetas `y` deben tener exactamente la **misma dimensión y ser de tipo `torch.float32`** (con valores `0.0` o `1.0`). Admite `pos_weight` para compensar el desbalanceo entre casos positivos y controles sanos.
- **`nn.MSELoss()` y `nn.L1Loss()` (Regresión continua):**
  - Empleadas para predecir variables biomédicas continuas (ej. edad biológica, concentración de glucosa, supervivencia o fracción de eyección).
  - `nn.MSELoss()` calcula el error cuadrático medio ($\frac{1}{N}\sum_{i=1}^N (\hat{y}_i - y_i)^2$), penalizando fuertemente las desviaciones grandes.
  - `nn.L1Loss()` calcula el error absoluto medio ($\frac{1}{N}\sum_{i=1}^N |\hat{y}_i - y_i|$), ofreciendo mayor robustez frente a valores atípicos (*outliers*) o ruido de medición clínica.

| Función de pérdida | Tarea | Salida esperada del modelo (`outputs`) | Formato requerido de etiquetas (`labels`) |
| :--- | :--- | :--- | :--- |
| **`nn.CrossEntropyLoss()`** | Clasificación multiclase ($C \ge 2$) | Logits crudos `[batch, C]` (`float32`, **sin Softmax**) | Índices de clase `[batch]` de tipo **`torch.long`** |
| **`nn.BCEWithLogitsLoss()`** | Binaria (1 nodo) o Multietiqueta | Logits crudos `[batch, 1]` o `[batch, K]` (**sin Sigmoid**) | Tensor `[batch, 1]` o `[batch, K]` de tipo **`torch.float32`** |
| **`nn.MSELoss()` / `nn.L1Loss()`** | Regresión continua | Valores continuos `[batch, 1]` (`float32`) | Valores continuos `[batch, 1]` de tipo **`torch.float32`** |

### Optimizadores (`torch.optim`)

- **`optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)`:**
  - Es el **estándar moderno por excelencia** tanto en modelos tabulares/convolucionales como en arquitecturas híbridas cuántico-clásicas.
  - A diferencia del `Adam` original (donde añadir penalización $L_2$ a la función de coste interfiere con las estimaciones de media y varianza móvil del gradiente $m_t$ y $v_t$), **AdamW desacopla la regularización $L_2$ (*weight decay*) del gradiente acumulado**, aplicando el decaimiento directamente sobre los pesos ($\theta_{t+1} = (1 - \eta \lambda)\theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$). Esto mejora sustancialmente la generalización.
- **`optim.SGD(model.parameters(), lr=1e-2, momentum=0.9, weight_decay=1e-4)`:**
  - Descenso de gradiente estocástico clásico acelerado con **momento** (que acumula una media exponencial de las direcciones previas para amortiguar oscilaciones y escapar de puntos de silla).
  - Aunque requiere un ajuste más cuidadoso de la tasa de aprendizaje inicial, suele converger hacia mínimos más planos con **mayor capacidad de generalización** en arquitecturas convolucionales clásicas (como familias ResNet o DenseNet).

### Planificadores de tasa de aprendizaje (`torch.optim.lr_scheduler`)

Mantener una tasa de aprendizaje (`lr`) fija durante todo el entrenamiento suele ser subóptimo: un `lr` alto al principio acelera el descenso rápido, pero impide asentarse con precisión en el mínimo local al final. Los *schedulers* ajustan dinámicamente el `lr` al finalizar cada época:

- **`ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=5)`:**
  - Monitoriza una métrica de validación (ej. `val_loss` con `mode='min'` o `val_auc` con `mode='max'`). Si la métrica deja de mejorar durante un número de épocas igual a `patience`, **reduce la tasa de aprendizaje multiplicándola por `factor`** ($\text{lr}_{\text{nuevo}} = \text{lr} \times 0.1$).
  - *Nota de uso:* requiere pasarle la métrica evaluada al final de la fase de validación: `scheduler.step(val_loss)`.
- **`CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)`:**
  - Reduce la tasa de aprendizaje de forma determinista y suave siguiendo el perfil de media curva de coseno desde el `lr` inicial hasta un valor mínimo `eta_min` a lo largo de `T_max` épocas:
    $$\eta_t = \eta_{\min} + \frac{1}{2}(\eta_{\max} - \eta_{\min})\left(1 + \cos\left(\frac{T_{\text{cur}}}{T_{\max}}\pi\right)\right)$$
  - *Nota de uso:* se actualiza al final de cada época sin argumentos: `scheduler.step()`.

In [ ]:
import torch.optim as optim

# 1. Demostración de las funciones de pérdida y sus tipos de datos requeridos
logits_multiclase = torch.tensor([[2.1, -1.0, 0.5], [-0.8, 1.9, 0.2]])  # [batch=2, num_classes=3] (SIN Softmax)
y_multiclase = torch.tensor([0, 1], dtype=torch.long)                   # [batch=2] entero torch.long

criterion_ce = nn.CrossEntropyLoss()
loss_ce = criterion_ce(logits_multiclase, y_multiclase)

logits_binarios = torch.tensor([[1.5], [-2.0]])                         # [batch=2, 1] (SIN Sigmoid)
y_binarios = torch.tensor([[1.0], [0.0]], dtype=torch.float32)          # [batch=2, 1] float32

criterion_bce = nn.BCEWithLogitsLoss()
loss_bce = criterion_bce(logits_binarios, y_binarios)

criterion_mse = nn.MSELoss()
criterion_l1 = nn.L1Loss()
pred_continua = torch.tensor([110.5, 145.0])
real_continua = torch.tensor([105.0, 150.0])

print(f"CrossEntropyLoss:  {loss_ce.item():.4f}")
print(f"BCEWithLogitsLoss: {loss_bce.item():.4f}")
print(f"MSELoss:           {criterion_mse(pred_continua, real_continua).item():.4f} | L1Loss: {criterion_l1(pred_continua, real_continua).item():.4f}")

# 2. Configuración de Optimizadores (AdamW vs SGD) y Schedulers de LR
model_demo = NeuralNetwork(in_features=10, num_classes=3)

optimizer_adamw = optim.AdamW(model_demo.parameters(), lr=1e-3, weight_decay=1e-2)
optimizer_sgd = optim.SGD(model_demo.parameters(), lr=1e-2, momentum=0.9)

scheduler_plateau = optim.lr_scheduler.ReduceLROnPlateau(optimizer_adamw, mode='min', factor=0.1, patience=2)
scheduler_cosine = optim.lr_scheduler.CosineAnnealingLR(optimizer_sgd, T_max=5, eta_min=1e-5)

# Simulación de evolución de la tasa de aprendizaje durante 5 épocas
val_losses_simuladas = [0.80, 0.82, 0.83, 0.84, 0.85]  # Pérdida estancada para activar ReduceLROnPlateau
for ep, v_loss in enumerate(val_losses_simuladas, start=1):
    scheduler_plateau.step(v_loss)
    scheduler_cosine.step()
    lr_adamw = optimizer_adamw.param_groups[0]['lr']
    lr_sgd = optimizer_sgd.param_groups[0]['lr']
    print(f"Época {ep} | val_loss={v_loss:.2f} -> LR AdamW (Plateau): {lr_adamw:.6f} | LR SGD (Cosine): {lr_sgd:.6f}")

---

## 1.6. El Pipeline Canónico de Entrenamiento y Validación

Para la memoria y el desarrollo experimental del TFG, estructurar correctamente el bucle canónico de entrenamiento y validación es crítico para garantizar la reproducibilidad científica y **evitar fugas de datos (*data leakage*)** entre los conjuntos de entrenamiento y evaluación.

### Anatomía del bucle canónico y prevención de *Data Leakage*

1. **Transferencia unificada al dispositivo (`device`):**
   - Tanto el modelo (`model.to(device)`) como cada lote de tensores (`inputs.to(device)`, `labels.to(device)`) deben residir en el mismo dispositivo (`cuda` o `cpu`).
2. **Fase de Entrenamiento (`model.train()`):**
   - Invocar `model.train()` al inicio de cada época **no entrena el modelo por sí mismo**, sino que conmuta las capas sensibles al modo de aprendizaje:
     - Activa **`Dropout`** (apagado estocástico de neuronas).
     - Configura **`BatchNorm`** para normalizar usando la media y varianza del lote actual y actualizar sus estadísticos poblacionales móviles (*running_mean* y *running_var*).
   - Por cada lote se ejecutan en orden estricto los **5 pasos canónicos**:
     1. `optimizer.zero_grad()`: limpiar los gradientes residuales de la iteración previa.
     2. `outputs = model(inputs)`: *forward pass* para obtener los logits.
     3. `loss = criterion(outputs, labels)`: calcular el valor escalar de la función de coste.
     4. `loss.backward()`: *backward pass* mediante Autograd para calcular $\nabla_{\boldsymbol{\theta}} \mathcal{L}$.
        - *(Opcional)* **Recorte de gradientes (*Gradient Clipping*):** `torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)` reescala el vector global de gradientes si su norma $L_2$ supera `max_norm`, evitando pasos descontrolados por **gradientes explosivos**.
     5. `optimizer.step()`: actualizar los pesos del modelo.
3. **Fase de Validación (`model.eval()` + `with torch.no_grad():`):**
   - **`model.eval()`:** congela las estadísticas móviles de `BatchNorm` (utilizando las aprendidas exclusivamente en entrenamiento, lo que **impide filtrar información estadística del conjunto de validación**) y desactiva `Dropout` para que la predicción sea determinista.
   - **`with torch.no_grad():`:** desactiva el motor Autograd durante toda la validación, reduciendo el uso de VRAM y acelerando la inferencia.
   - **Acumulación ponderada de pérdidas:** multiplicar `loss.item() * inputs.size(0)` pondera exactamente la pérdida media de cada lote por su número real de pacientes, evitando sesgos cuando el último lote del `DataLoader` tiene un tamaño menor que `batch_size`.
   - **Actualización del *scheduler*:** al concluir la época completa de validación se ejecuta `scheduler.step(val_loss)`.

In [ ]:
from torch.utils.data import TensorDataset

# Preparación de datos sintéticos biomédicos (sin data leakage: partición separada train / val)
torch.manual_seed(42)
X_train_sim = torch.randn(64, 10)
y_train_sim = (X_train_sim[:, 0] + X_train_sim[:, 1] > 0).long()
X_val_sim = torch.randn(24, 10)
y_val_sim = (X_val_sim[:, 0] + X_val_sim[:, 1] > 0).long()

train_loader = DataLoader(TensorDataset(X_train_sim, y_train_sim), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_sim, y_val_sim), batch_size=16, shuffle=False)

model = NeuralNetwork(in_features=10, num_classes=2)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-2, weight_decay=1e-2)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.1, patience=3)
num_epochs = 5

# --- PIPELINE CANÓNICO DE ENTRENAMIENTO Y VALIDACIÓN ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(num_epochs):
    # --- FASE DE ENTRENAMIENTO ---
    model.train()  # Activa Dropout y BatchNorm en modo aprendizaje
    train_loss = 0.0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()             # 1. Limpiar gradientes anteriores
        outputs = model(inputs)           # 2. Forward pass (logits)
        loss = criterion(outputs, labels) # 3. Calcular función de pérdida
        loss.backward()                   # 4. Backward pass (calcular dLoss/dw)

        # Opcional: Gradient clipping para evitar gradientes explosivos
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()                  # 5. Actualización de pesos

        train_loss += loss.item() * inputs.size(0)

    # --- FASE DE VALIDACIÓN ---
    model.eval()  # Congela BatchNorm y desactiva Dropout
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():  # Deshabilita Autograd para ahorrar memoria y tiempo
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(outputs, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    # Promedio por muestra y actualización del scheduler al final de la época
    train_loss /= len(train_loader.dataset)
    val_loss /= len(val_loader.dataset)
    val_acc = correct / total

    scheduler.step(val_loss)
    print(f"Época [{epoch+1}/{num_epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.1f}%")

---

## 1.7. Persistencia, Checkpointing y Buenas Prácticas de GPU

### Guardado y carga de modelos (`torch.save` y `torch.load`)

En PyTorch, el estado interno de un modelo o de un optimizador se almacena en un diccionario de Python llamado **`state_dict`**, que mapea cada capa a sus tensores de parámetros (`weight`, `bias`) y *buffers* persistentes (como `running_mean` de `BatchNorm`). Existen dos estrategias principales de persistencia:

1. **Solo pesos (`state_dict`) — Práctica estándar recomendada para inferencia y despliegue:**
   - Guarda únicamente el diccionario de parámetros del modelo (extensión `.pt` o `.pth`), evitando serializar la clase completa con `pickle` (lo que acoplaría el archivo a la estructura exacta de carpetas y código fuente).
   - **Uso de `map_location=device`:** resulta indispensable al cargar pesos en un entorno distinto al de entrenamiento (por ejemplo, entrenar el modelo en un servidor con GPU `cuda:0` y cargarlo luego en un portátil con `cpu`).
   ```python
   torch.save(model.state_dict(), "best_model.pt")
   model.load_state_dict(torch.load("best_model.pt", map_location=device))
   model.eval()  # Recordar fijar modo evaluación antes de inferir
   ```

2. **Checkpoint completo — Para reanudar el entrenamiento tras interrupciones:**
   - Cuando los entrenamientos duran horas o días (habitual en volúmenes médicos 3D o simulaciones de circuitos cuánticos variacionales), guardar solo los pesos del modelo es insuficiente porque optimizadores como `AdamW` o `SGD` con momento mantienen buffers internos de primer y segundo momento para cada parámetro.
   - Un *checkpoint* completo (habitualmente con extensión `.tar` o `.pt`) empaqueta en un diccionario la época actual (`epoch`), los pesos del modelo (`model_state_dict`), el estado del optimizador (`optimizer_state_dict`), el estado del scheduler y la pérdida de validación (`loss`):
   ```python
   torch.save({
       'epoch': epoch,
       'model_state_dict': model.state_dict(),
       'optimizer_state_dict': optimizer.state_dict(),
       'loss': val_loss,
   }, "checkpoint.tar")
   ```

### Gestión de memoria VRAM y prevención de fugas (*Memory Leaks*)

- **Inspección del hardware GPU:**
  - `torch.cuda.is_available()`: devuelve `True` si hay una GPU compatible con CUDA accesible.
  - `torch.cuda.get_device_name(0)`: devuelve el nombre comercial de la GPU (ej. `NVIDIA RTX 4090`).
  - `torch.cuda.memory_allocated()` y `torch.cuda.memory_reserved()`: informan respectivamente de la VRAM ocupada actualmente por tensores y del total reservado en caché por el asignador de PyTorch.
- **`torch.cuda.empty_cache()`:**
  - PyTorch utiliza un asignador de memoria con caché (*caching allocator*) para evitar lentas llamadas al sistema CUDA cada vez que se crea o destruye un tensor. `torch.cuda.empty_cache()` **libera todos los bloques de memoria reservados pero no asignados activamente** que retiene dicho asignador, devolviéndolos a la GPU. Resulta muy útil al capturar excepciones de falta de memoria (`torch.cuda.OutOfMemoryError`) o al limpiar recursos entre distintos pliegues (*folds*) de una validación cruzada.
- **Regla crítica: usar `.item()` al extraer escalares de pérdida en bucles (`loss.item()`):**
  - El tensor `loss` devuelto por la función de coste está conectado a **todo el grafo computacional (DAG)** de esa iteración a través de su `.grad_fn`.
  - Si en el bucle de entrenamiento o validación se escribe `total_loss += loss` (sin `.item()`), Python no suma un número flotante, sino que **encadena en un grafo creciente los grafos computacionales completos de todas las iteraciones**, impidiendo que el recolector de basura libere las activaciones intermedias y provocando una **fuga masiva de RAM/VRAM (*memory leak*)** en pocas iteraciones.
  - `loss.item()` extrae únicamente el número escalar nativo de Python (`float`), desligándolo por completo del grafo.

### Precisión Mixta Automática (AMP — *Automatic Mixed Precision*)

Por defecto, PyTorch opera en precisión simple de 32 bits (`float32` / FP32). Las GPUs modernas incorporan unidades de hardware especializadas (*Tensor Cores*) capaces de realizar álgebra matricial en media precisión de 16 bits (`float16` / FP16 o `bfloat16`) a una velocidad muy superior:

- **`torch.amp.autocast('cuda')`:** gestor de contexto que envuelve el *forward pass* y el cálculo de la función de pérdida. Selecciona automáticamente la precisión óptima para cada operación:
  - Ejecuta en **`float16`** las operaciones intensivas en cómputo matricial (`nn.Linear`, `nn.Conv2d`, `nn.Conv3d`, `torch.matmul`).
  - Mantiene en **`float32`** las operaciones numéricamente sensibles a errores de redondeo o desbordamiento (funciones de pérdida como `CrossEntropyLoss`, reducciones `.sum()`/`.mean()`, capas de normalización `BatchNorm`/`LayerNorm` y exponenciales/`Softmax`).
- **`torch.cuda.amp.GradScaler()` (o `torch.amp.GradScaler('cuda')`):**
  - En formato `float16`, los gradientes de magnitud muy pequeña ($< 6 \times 10^{-8}$) sufren **subdesbordamiento (*underflow*)**, convirtiéndose en `0.0` e impidiendo que el modelo aprenda.
  - `GradScaler` resuelve este problema **multiplicando la pérdida por un factor de escala grande** antes de llamar a `.backward()` (`scaler.scale(loss).backward()`), desplazando así todos los gradientes hacia un rango numérico seguro en `float16`. Posteriormente, antes de actualizar los pesos, `scaler.step(optimizer)` desescala los gradientes (y salta la actualización si detecta algún `Inf` o `NaN`), mientras que `scaler.update()` ajusta dinámicamente el factor de escala para la siguiente iteración.
- **Impacto práctico:** acelera **hasta el doble ($2\times$) la velocidad de entrenamiento** y **reduce aproximadamente a la mitad ($50\%$) el consumo de memoria VRAM** en GPUs modernas, permitiendo duplicar el `batch_size` o entrenar con volúmenes médicos de mayor resolución.

In [ ]:
import io

# 1. Guardado y carga de solo pesos (state_dict) vs Checkpoint completo
# (Usamos un buffer en memoria BytesIO para la demostración sin dejar archivos residuales en disco;
#  en tu código reemplaza buffer_pesos / buffer_ckpt por "best_model.pt" y "checkpoint.tar")
buffer_pesos = io.BytesIO()
torch.save(model.state_dict(), buffer_pesos)  # Equivalente a: torch.save(model.state_dict(), "best_model.pt")

buffer_pesos.seek(0)
model.load_state_dict(torch.load(buffer_pesos, map_location=device, weights_only=True))
model.eval()
print("Pesos cargados correctamente desde state_dict en dispositivo:", device)

# Guardado y restauración de un Checkpoint completo para reanudar entrenamiento
buffer_ckpt = io.BytesIO()
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'loss': val_loss,
}, buffer_ckpt)  # Equivalente a: torch.save({...}, "checkpoint.tar")

buffer_ckpt.seek(0)
checkpoint = torch.load(buffer_ckpt, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
epoca_reanudada = checkpoint['epoch']
loss_registrada = checkpoint['loss']
print(f"Checkpoint restaurado -> Época: {epoca_reanudada + 1} | Val Loss previa: {loss_registrada:.4f}")

# 2. Inspección de GPU y limpieza de caché de VRAM
if torch.cuda.is_available():
    print("GPU detectada:", torch.cuda.get_device_name(0))
    torch.cuda.empty_cache()
else:
    print("Ejecutando en CPU (torch.cuda.is_available() == False)")

# 3. Bucle de entrenamiento con Precisión Mixta Automática (AMP: autocast + GradScaler)
use_amp = torch.cuda.is_available()
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

model.train()
for inputs, labels in train_loader:
    inputs, labels = inputs.to(device), labels.to(device)
    optimizer.zero_grad()

    # Forward pass en precisión mixta (float16 en operaciones matriciales / float32 en pérdida)
    with torch.amp.autocast(device_type=device.type, enabled=use_amp):
        outputs = model(inputs)
        loss = criterion(outputs, labels)

    # Escalado de pérdida para evitar subdesbordamiento (underflow) de gradientes en float16
    scaler.scale(loss).backward()

    # Si se usa gradient clipping con AMP, primero se desescalan explícitamente los gradientes:
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # Actualización de pesos y del factor de escala
    scaler.step(optimizer)
    scaler.update()

print(f"Iteración con AMP completada correctamente | Última pérdida (.item()): {loss.item():.4f}")